## Scenes and Automatization 1

In [1]:
# =========================================
# CONFIG
# =========================================
YEARS            = list(range(2017, 2018))
MONTH_START_END  = ("07-01", "08-31")
GRID             = "MGRS-05WMU"
MAX_CLOUD_COVER  = 30
OUT_DIR          = "CDSE_yearly_median_masked/coverage70"

import os
os.environ["AWS_ACCESS_KEY_ID"]     = "C364NPCJK6JQ64OIMZJR"
os.environ["AWS_SECRET_ACCESS_KEY"] = "Jl0uXRVeaPGsJoWcR7510zIvlV7utzuffWb8rkdh"   # fill in securely
os.environ["AWS_REGION"]            = "us-east-1"
os.environ["AWS_S3_ENDPOINT"]       = "eodata.dataspace.copernicus.eu"
os.environ["AWS_VIRTUAL_HOSTING"]   = "FALSE"

# =========================================
# IMPORTS
# =========================================
from pathlib import Path
import math
import numpy as np
import xarray as xr
import rioxarray
import rasterio as rio
from rasterio.enums import Resampling
from pyproj import Transformer  # only used in EPSG fallback
from pystac_client import Client
from pystac import Item
from omnicloudmask import predict_from_array

# =========================================
# HELPERS
# =========================================
def search_s2_stac(start_date: str, end_date: str, grid: str, max_cloud_cover: int = 100) -> list[Item]:
    """Search Sentinel-2 L2A STAC for a given grid tile and time range."""
    cat = Client.open("https://stac.dataspace.copernicus.eu/v1/")
    search = cat.search(
        collections=["sentinel-2-l2a"],
        datetime=f"{start_date}/{end_date}",
        query={
            "eo:cloud_cover": {"lte": max_cloud_cover},
            "grid:code": {"eq": grid},
        },
    )
    items = list(search.items())
    print(f"  🔎 Found {len(items)} items")
    return items

def prefer_s3_assets(items):
    """Switch assets to S3 hrefs when available."""
    out = []
    for it in items:
        it = it.clone()
        for a in it.assets.values():
            s3_href = None
            extra = (getattr(a, "extra_fields", None) or {})
            alt = extra.get("alternate") or extra.get("alternates")
            if isinstance(alt, dict):
                s3_href = (alt.get("s3") or alt.get("S3") or {}).get("href")
            elif isinstance(alt, list):
                for d in alt:
                    href = d.get("href")
                    if href and href.startswith("s3://"):
                        s3_href = href
                        break
            if s3_href:
                a.href = s3_href
        out.append(it)
    return out

def detect_epsg(items):
    """Get EPSG from STAC items or fall back using centroid longitude/latitude."""
    for it in items:
        if "proj:epsg" in it.properties:
            epsg = int(it.properties["proj:epsg"])
            print(f"  EPSG from metadata: {epsg}")
            return epsg

    # Fallback: use bbox center of first item
    it0 = items[0]
    minx, miny, maxx, maxy = it0.bbox
    lon = (minx + maxx) / 2.0
    lat = (miny + maxy) / 2.0
    zone = int(math.floor((lon + 180) / 6) + 1)
    epsg = 32600 + zone if lat >= 0 else 32700 + zone
    print(f"  EPSG fallback (UTM): {epsg}")
    return epsg

def rasterio_env():
    """Common rasterio GDAL environment to talk to CDSE S3."""
    return rio.Env(
        AWS_S3_ENDPOINT=os.environ["AWS_S3_ENDPOINT"],
        AWS_REGION=os.environ["AWS_REGION"],
        AWS_VIRTUAL_HOSTING=os.environ["AWS_VIRTUAL_HOSTING"],
        GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
        CPL_VSIL_CURL_ALLOWED_EXTENSIONS="tif,gtiff,jp2,xml"
    )

# =========================================
# PROCESS
# =========================================
BAND_ORDER  = ["B02_10m","B03_10m","B04_10m","B08_10m","B11_20m","B12_20m"]
BAND_LABELS = ["Blue","Green","Red","NIR","SWIR1","SWIR2"]

def build_reference_grid(ref_item_s3, epsg_out: int):
    """
    Build a reference grid (CRS, transform, resolution, shape) for the tile
    from the first available 10m band.
    """
    with rasterio_env():
        # Prefer any 10m band as template
        ref_band_name = None
        for bname in BAND_ORDER:
            if bname.endswith("10m") and bname in ref_item_s3.assets:
                ref_band_name = bname
                break

        if ref_band_name is None:
            raise RuntimeError("No 10m band found in reference item.")

        href = ref_item_s3.assets[ref_band_name].href
        da = rioxarray.open_rasterio(href, masked=True).squeeze("band", drop=True)
        if da.rio.crs is None:
            da = da.rio.write_crs(f"EPSG:{epsg_out}")
        print(f"  Reference grid from {ref_item_s3.id} ({ref_band_name})")
        return da

def load_and_mask_scene(it_s3, it_orig, template_da, epsg_out: int):
    """
    Load full scene for a single STAC item:
      - read all bands in BAND_ORDER
      - reproject to template grid
      - apply omnicloudmask
      - return masked xarray.DataArray with dims (band, y, x)
    """
    bands_10m = [b for b in BAND_ORDER if b.endswith("10m")]
    bands_20m = [b for b in BAND_ORDER if b.endswith("20m")]

    with rasterio_env():
        pieces = []

        # 10m bands
        for bname in bands_10m:
            if bname not in it_s3.assets:
                continue
            href = it_s3.assets[bname].href
            da = rioxarray.open_rasterio(href, masked=True).squeeze("band", drop=True)
            if da.rio.crs is None:
                da = da.rio.write_crs(f"EPSG:{epsg_out}")
            # Reproject to template grid (full tile)
            da = da.rio.reproject_match(template_da, resampling=Resampling.bilinear)
            pieces.append(da.expand_dims("band"))

        # 20m bands → upsample to 10m template
        for bname in bands_20m:
            if bname not in it_s3.assets:
                continue
            href = it_s3.assets[bname].href
            da20 = rioxarray.open_rasterio(href, masked=True).squeeze("band", drop=True)
            if da20.rio.crs is None:
                da20 = da20.rio.write_crs(f"EPSG:{epsg_out}")
            da20u = da20.rio.reproject_match(template_da, resampling=Resampling.bilinear)
            pieces.append(da20u.expand_dims("band"))

    if not pieces:
        raise RuntimeError("No usable bands found for scene.")

    scene = xr.concat(pieces, dim="band")
    scene = scene.assign_coords(band=BAND_LABELS)
    if scene.rio.crs is None:
        scene = scene.rio.write_crs(f"EPSG:{epsg_out}")

    # ------------- CLOUD MASKING (full tile) -------------
    red   = scene.sel(band="Red").values
    green = scene.sel(band="Green").values
    nir   = scene.sel(band="NIR").values
    input_array = np.stack([red, green, nir], axis=0)

    pred_mask = predict_from_array(input_array)

    # Handle shape (1, H, W) or (3, H, W)
    if pred_mask.ndim == 3:
        if pred_mask.shape[0] == 1:
            pred_mask = pred_mask[0]
        elif pred_mask.shape[0] == 3:
            # assuming class index 1 = cloud
            pred_mask = pred_mask[1]

    # Ensure mask shape matches (y, x)
    if pred_mask.shape != (scene.sizes["y"], scene.sizes["x"]):
        raise ValueError(
            f"Mask shape {pred_mask.shape} does not match scene shape "
            f"({scene.sizes['y']}, {scene.sizes['x']})"
        )

    # Keep only pixels where class == 0 (clear)
    mask_keep = pred_mask == 0
    mask_da = xr.DataArray(
        mask_keep,
        dims=("y", "x"),
        coords={"y": scene.coords["y"], "x": scene.coords["x"]},
    )

    scene_masked = scene.where(mask_da)

    return scene_masked

def process_year(year: int, grid: str, max_cloud: int):
    """
    For a given year:
      * search all Sentinel-2 L2A scenes of that grid tile in the date range
      * load full scenes, mask with OmniCloudMask
      * stack them and compute median per pixel
      * save a single median image for the year
    """
    print(f"\n==== Year {year} | grid={grid} | clouds≤{max_cloud}% ====")
    start_date = f"{year}-{MONTH_START_END[0]}"
    end_date   = f"{year}-{MONTH_START_END[1]}"

    items = search_s2_stac(start_date, end_date, grid, max_cloud_cover=max_cloud)
    if not items:
        print("  ⚠️ No items for this year.")
        return 0

    items_s3 = prefer_s3_assets(items)
    epsg_out = detect_epsg(items)

    # Build template grid from first item (full tile extent)
    template_da = build_reference_grid(items_s3[0], epsg_out)

    out_dir = Path(OUT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)

    scenes = []
    n_ok = 0

    for it_s3, it_orig in zip(items_s3, items):
        scene_date = it_orig.properties.get("datetime", "").split("T")[0]
        print(f"\n→ Scene {it_orig.id} ({scene_date})")

        try:
            scene_masked = load_and_mask_scene(it_s3, it_orig, template_da, epsg_out)
            scenes.append(scene_masked.expand_dims("scene"))
            n_ok += 1
            print("   ✔ Scene loaded & masked.")
        except Exception as e:
            print("   ❌ Scene failed:", e)

    if not scenes:
        print("  ⚠️ No valid scenes to aggregate for this year.")
        return 0

    # =========================================
    # YEARLY MEDIAN
    # =========================================
    print(f"\n📊 Computing median of {n_ok} scenes for {year} ...")
    stack = xr.concat(scenes, dim="scene")

    # median per pixel, ignoring NaNs (clouds)
    median_scene = stack.median(dim="scene", skipna=True)

    median_u16 = (
        median_scene.fillna(0)
        .clip(0, 10000)
        .astype("uint16")
        .rio.write_nodata(0)
        .rio.write_crs(f"EPSG:{epsg_out}")
    )

    out_path = out_dir / f"{grid}_{year}_median_masked.tif"
    print("   💾 Saving yearly median →", out_path)

    median_u16.transpose("band", "y", "x").rio.to_raster(
        out_path,
        driver="GTiff",
        compress="deflate",
        tiled=True,
        predictor=2,
        BIGTIFF="IF_SAFER",
        blockxsize=512,
        blockysize=512,
    )

    return n_ok

# =========================================
# RUN ALL YEARS
# =========================================
all_counts = {}
for yr in YEARS:
    n = process_year(yr, GRID, MAX_CLOUD_COVER)
    all_counts[yr] = n

print("\n✅ Done. Scenes used per year (for median):")
for yr, n in all_counts.items():
    print(f" • {yr}: {n} scenes")


/home/pd/sipohl001/miniconda3/envs/env_omnicloud/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



==== Year 2017 | grid=MGRS-05WMU | clouds≤30% ====
  🔎 Found 8 items
  EPSG fallback (UTM): 32605
  Reference grid from S2A_MSIL2A_20170728T224531_N0500_R101_T05WMU_20230903T171903 (B02_10m)

→ Scene S2A_MSIL2A_20170728T224531_N0500_R101_T05WMU_20230903T171903 (2017-07-28)


KeyboardInterrupt: 

## Median automization

In [ ]:
# =========================================
# CONFIG
# =========================================
YEARS            = list(range(2017, 2026))
MONTH_START_END  = ("07-01", "08-31")
GRID             = "MGRS-05WMN" #"MGRS-05WMU"
MAX_CLOUD_COVER  = 70
BBOX_LL          = (-154.25, 65.00, -154.00, 65.25) #(-153.5, 70.5, -153, 71)
OUT_DIR          = "CDSE_yearly_medians"


import os
os.makedirs(OUT_DIR, exist_ok=True)
os.environ["AWS_ACCESS_KEY_ID"]     = "C364NPCJK6JQ64OIMZJR"
os.environ["AWS_SECRET_ACCESS_KEY"] = "Jl0uXRVeaPGsJoWcR7510zIvlV7utzuffWb8rkdh"  
os.environ["AWS_REGION"]            = "us-east-1"
os.environ["AWS_S3_ENDPOINT"]       = "eodata.dataspace.copernicus.eu"
os.environ["AWS_VIRTUAL_HOSTING"]   = "FALSE"

# =========================================
# IMPORTS
# =========================================
from pathlib import Path
import math
import numpy as np
import xarray as xr
import rioxarray
from pyproj import Transformer
import rasterio as rio
from rasterio.enums import Resampling
from pystac_client import Client
from shapely.geometry import shape, box
from shapely.ops import transform as shapely_transform
from omnicloudmask import predict_from_array
from pystac_client import Client
from pystac import Item

# =========================================
# HELPERS (unchanged)
# =========================================
def search_s2_stac(start_date: str, end_date: str, grid: str, max_cloud_cover: int = 100) -> list[Item]:
    cat = Client.open("https://stac.dataspace.copernicus.eu/v1/")
    search = cat.search(
        collections=["sentinel-2-l2a"],
        datetime=f"{start_date}/{end_date}",
        query={"eo:cloud_cover": {"lte": max_cloud_cover}, "grid:code": {"eq": grid}},
    )
    items = list(search.items())
    print(f"  🔎 Found {len(items)} items")
    return items

def prefer_s3_assets(items):
    out = []
    for it in items:
        it = it.clone()
        for a in it.assets.values():
            s3_href = None
            extra = (getattr(a, "extra_fields", None) or {})
            alt = extra.get("alternate") or extra.get("alternates")
            if isinstance(alt, dict):
                s3_href = (alt.get("s3") or alt.get("S3") or {}).get("href")
            elif isinstance(alt, list):
                for d in alt:
                    href = d.get("href")
                    if href and href.startswith("s3://"):
                        s3_href = href
                        break
            if s3_href:
                a.href = s3_href
        out.append(it)
    return out

def detect_epsg_and_bounds(items, bbox_ll_override=None):
    if not items:
        raise ValueError("No items")

    if bbox_ll_override is None:
        bbs = [it.bbox for it in items]
        minx = min(b[0] for b in bbs)
        miny = min(b[1] for b in bbs)
        maxx = max(b[2] for b in bbs)
        maxy = max(b[3] for b in bbs)
        bbox_ll = (minx, miny, maxx, maxy)
    else:
        bbox_ll = bbox_ll_override

    epsg = None
    for it in items:
        if "proj:epsg" in it.properties:
            epsg = int(it.properties["proj:epsg"])
            break
    if epsg is None:
        lon = (bbox_ll[0] + bbox_ll[2]) / 2.0
        lat = (bbox_ll[1] + bbox_ll[3]) / 2.0
        zone = int(math.floor((lon + 180) / 6) + 1)
        epsg = 32600 + zone if lat >= 0 else 32700 + zone

    tx = Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)
    x1, y1 = tx.transform(bbox_ll[0], bbox_ll[1])
    x2, y2 = tx.transform(bbox_ll[2], bbox_ll[3])
    bounds_proj = (min(x1, x2), min(y1, y2), max(x1, x2), max(y1, y2))
    return epsg, bbox_ll, bounds_proj

def projected_intersection_ratio(item_geom, aoi_bounds, epsg_out):
    # Transform AOI bbox (in lon/lat) to projected coords
    tx = Transformer.from_crs("EPSG:4326", f"EPSG:{epsg_out}", always_xy=True)
    aoi_proj = shapely_transform(tx.transform, box(*aoi_bounds))

    # Get item's footprint and project it too
    geom = shape(item_geom)
    geom_proj = shapely_transform(tx.transform, geom)

    inter = geom_proj.intersection(aoi_proj)

    if inter.is_empty:
        return 0.0

    return inter.area / aoi_proj.area

def rasterio_env():
    return rio.Env(
        AWS_S3_ENDPOINT=os.environ["AWS_S3_ENDPOINT"],
        AWS_REGION=os.environ["AWS_REGION"],
        AWS_VIRTUAL_HOSTING=os.environ["AWS_VIRTUAL_HOSTING"],
        GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
        CPL_VSIL_CURL_ALLOWED_EXTENSIONS="tif,gtiff,jp2,xml"
    )


# =========================================
# PROCESS YEAR → RETURN MEDIAN ARRAY
# =========================================
BAND_ORDER  = ["B02_10m","B03_10m","B04_10m","B08_10m","B11_20m","B12_20m"]
BAND_LABELS = ["Blue","Green","Red","NIR","SWIR1","SWIR2"]

def process_year_to_median(year: int, grid: str, max_cloud: int, bbox_ll):

    print(f"\n==== YEAR {year} | {grid} | clouds ≤ {max_cloud}% ====")
    start_date = f"{year}-{MONTH_START_END[0]}"
    end_date   = f"{year}-{MONTH_START_END[1]}"

    # Search for scenes
    items = search_s2_stac(start_date, end_date, grid, max_cloud_cover=max_cloud)
    if not items:
        return None

    items_s3 = prefer_s3_assets(items)
    epsg_out, bbox_ll_used, bounds_out = detect_epsg_and_bounds(items, bbox_ll_override=bbox_ll)

    # Temporary list to accumulate scenes in memory
    scene_stack = []

    bands_10m = [b for b in BAND_ORDER if b.endswith("10m")]
    bands_20m = [b for b in BAND_ORDER if b.endswith("20m")]

    with rasterio_env():
        for it_s3, it_orig in zip(items_s3, items):

            scene_date = it_orig.properties.get("datetime", "").split("T")[0]
            print(f"\n→ {it_orig.id} ({scene_date})")

            # Skip if AOI intersection is too small
            coverage_ratio = projected_intersection_ratio(
                it_orig.geometry, bbox_ll, epsg_out
            )
            print(f"   AOI coverage: {coverage_ratio:.2%}")
            if coverage_ratio < 0.4:
                print("   ⚠ Skipped")
                continue

            try:
                ref = None
                pieces = []

                # Load 10m bands
                for bname in bands_10m:
                    if bname not in it_s3.assets:
                        continue
                    da = rioxarray.open_rasterio(it_s3.assets[bname].href, masked=True).squeeze()
                    if da.rio.crs is None:
                        da = da.rio.write_crs(f"EPSG:{epsg_out}")
                    da = da.rio.clip_box(*bounds_out)
                    if ref is None:
                        ref = da
                    pieces.append(da.expand_dims("band"))

                # Load + upsample 20m bands
                for bname in bands_20m:
                    if bname not in it_s3.assets:
                        continue
                    da20 = rioxarray.open_rasterio(it_s3.assets[bname].href, masked=True).squeeze()
                    if da20.rio.crs is None:
                        da20 = da20.rio.write_crs(f"EPSG:{epsg_out}")
                    da20 = da20.rio.clip_box(*bounds_out)
                    da20u = da20.rio.reproject_match(ref, resampling=Resampling.bilinear)
                    pieces.append(da20u.expand_dims("band"))

                scene = xr.concat(pieces, dim="band")
                scene = scene.assign_coords(band=BAND_LABELS)

                # CLOUD MASK
                try:
                    red = scene.sel(band="Red").values
                    green = scene.sel(band="Green").values
                    nir = scene.sel(band="NIR").values

                    inp = np.stack([red, green, nir], axis=0)
                    pred_mask = predict_from_array(inp)

                    # Normalize shape
                    if pred_mask.ndim == 3:
                        pred_mask = pred_mask[1] if pred_mask.shape[0] == 3 else pred_mask[0]

                    mask_keep = pred_mask == 0
                    scene = scene.where(mask_keep)
                except Exception as e:
                    print("   Cloud mask failed:", e)

                scene_stack.append(scene)

            except Exception as e:
                print("   Scene failed:", e)
                continue

    if not scene_stack:
        print("⚠ No valid scenes for year", year)
        return None

    # ---------------------------------------
    # STACK ALL SCENES → MEDIAN
    # ---------------------------------------
    print(f"📊 Computing median for {len(scene_stack)} scenes…")
    stack = xr.concat(scene_stack, dim="time")
    median_img = stack.median(dim="time", skipna=True)

    return median_img, epsg_out


# =========================================
# RUN YEARS + WRITE MEDIAN TIFFS
# =========================================
for yr in YEARS:
    result = process_year_to_median(yr, GRID, MAX_CLOUD_COVER, bbox_ll=BBOX_LL)
    if result is None:
        continue

    median_img, epsg_out = result
    out_path = Path(OUT_DIR) / f"median_{yr}.tif"

    median_u16 = (
        median_img
        .fillna(0)
        .clip(0, 10000)
        .astype("uint16")
        .rio.write_nodata(0)
        .rio.write_crs(f"EPSG:{epsg_out}")
    )

    print(f"💾 Saving yearly median → {out_path}")
    median_u16.transpose("band", "y", "x").rio.to_raster(
        out_path,
        driver="GTiff",
        compress="deflate",
        tiled=True,
        predictor=2,
        BIGTIFF="IF_SAFER",
        blockxsize=512,
        blockysize=512,
    )

print("\n✅ All yearly medians written.")


/home/pd/sipohl001/miniconda3/envs/env_omnicloud/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



==== YEAR 2017 | MGRS-05WMN | clouds ≤ 70% ====
  🔎 Found 9 items

→ S2B_MSIL2A_20170820T220529_N0500_R072_T05WMN_20230826T215515 (2017-08-20)
   AOI coverage: 100.00%

→ S2B_MSIL2A_20170807T215529_N0500_R029_T05WMN_20230901T164019 (2017-08-07)
   AOI coverage: 100.00%

→ S2A_MSIL2A_20170805T220531_N0500_R072_T05WMN_20230826T060514 (2017-08-05)
   AOI coverage: 100.00%

→ S2A_MSIL2A_20170730T214531_N0500_R129_T05WMN_20231005T133800 (2017-07-30)
   AOI coverage: 0.00%
   ⚠ Skipped

→ S2A_MSIL2A_20170729T221531_N0500_R115_T05WMN_20230913T030334 (2017-07-29)
   AOI coverage: 0.00%
   ⚠ Skipped

→ S2B_MSIL2A_20170718T215529_N0500_R029_T05WMN_20230919T191859 (2017-07-18)
   AOI coverage: 100.00%

→ S2A_MSIL2A_20170716T220531_N0500_R072_T05WMN_20230911T152128 (2017-07-16)
   AOI coverage: 100.00%

→ S2B_MSIL2A_20170715T214529_N0500_R129_T05WMN_20230901T031916 (2017-07-15)
   AOI coverage: 0.00%
   ⚠ Skipped

→ S2B_MSIL2A_20170704T221529_N0500_R115_T05WMN_20230912T025341 (2017-07-04)
   AOI 

ERROR 1: PROJ: internal_proj_create_from_database: /home/pd/sipohl001/miniconda3/envs/env_omnicloud/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 3 is expected. It comes from another PROJ installation.


💾 Saving yearly median → CDSE_yearly_medians/median_2017.tif

==== YEAR 2018 | MGRS-05WMN | clouds ≤ 70% ====
  🔎 Found 19 items

→ S2B_MSIL2A_20180818T221529_N0500_R115_T05WMN_20230711T161247 (2018-08-18)
   AOI coverage: 0.00%
   ⚠ Skipped

→ S2A_MSIL2A_20180810T220531_N0500_R072_T05WMN_20230814T003508 (2018-08-10)
   AOI coverage: 100.00%

→ S2B_MSIL2A_20180809T214529_N0500_R129_T05WMN_20230711T054310 (2018-08-09)
   AOI coverage: 0.00%
   ⚠ Skipped

→ S2B_MSIL2A_20180808T221529_N0500_R115_T05WMN_20230826T085402 (2018-08-08)
   AOI coverage: 0.00%
   ⚠ Skipped

→ S2B_MSIL2A_20180808T221529_N0500_R115_T05WMN_20230624T082416 (2018-08-08)
   AOI coverage: 0.00%
   ⚠ Skipped

→ S2A_MSIL2A_20180807T215531_N0500_R029_T05WMN_20230708T234516 (2018-08-07)
   AOI coverage: 100.00%

→ S2A_MSIL2A_20180731T220531_N0500_R072_T05WMN_20230628T134851 (2018-07-31)
   AOI coverage: 100.00%

→ S2B_MSIL2A_20180729T221529_N0500_R115_T05WMN_20230723T140944 (2018-07-29)
   AOI coverage: 0.00%
   ⚠ Skipped


ERROR 1: PROJ: internal_proj_create_from_database: /home/pd/sipohl001/miniconda3/envs/env_omnicloud/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 3 is expected. It comes from another PROJ installation.


...

💾 Saving yearly median → CDSE_yearly_medians/median_2022.tif

==== YEAR 2023 | MGRS-05WMN | clouds ≤ 70% ====
  🔎 Found 16 items

→ S2A_MSIL2A_20230828T214531_N0510_R129_T05WMN_20241030T002628 (2023-08-28)
   AOI coverage: 0.00%
   ⚠ Skipped

→ S2A_MSIL2A_20230827T221541_N0510_R115_T05WMN_20241014T172214 (2023-08-27)
   AOI coverage: 0.00%
   ⚠ Skipped

→ S2B_MSIL2A_20230816T215539_N0510_R029_T05WMN_20241016T120507 (2023-08-16)
   AOI coverage: 100.00%

→ S2A_MSIL2A_20230811T215531_N0510_R029_T05WMN_20241029T182229 (2023-08-11)
   AOI coverage: 100.00%

→ S2B_MSIL2A_20230806T215539_N0510_R029_T05WMN_20241121T010323 (2023-08-06)
   AOI coverage: 100.00%

→ S2A_MSIL2A_20230804T220531_N0510_R072_T05WMN_20240823T160932 (2023-08-04)
   AOI coverage: 100.00%

→ S2A_MSIL2A_20230801T215531_N0510_R029_T05WMN_20241030T032814 (2023-08-01)
   AOI coverage: 100.00%

→ S2B_MSIL2A_20230730T220539_N0510_R072_T05WMN_20241014T124522 (2023-07-30)
   AOI coverage: 100.00%

→ S2A_MSIL2A_20230725T220531_N05

ERROR 1: PROJ: internal_proj_create_from_database: /home/pd/sipohl001/miniconda3/envs/env_omnicloud/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 3 is expected. It comes from another PROJ installation.


💾 Saving yearly median → CDSE_yearly_medians/median_2023.tif

==== YEAR 2024 | MGRS-05WMN | clouds ≤ 70% ====
  🔎 Found 12 items

→ S2A_MSIL2A_20240831T221531_N0511_R115_T05WMN_20240901T012951 (2024-08-31)
   AOI coverage: 0.00%
   ⚠ Skipped

→ S2A_MSIL2A_20240828T220531_N0511_R072_T05WMN_20240829T010711 (2024-08-28)
   AOI coverage: 100.00%

→ S2B_MSIL2A_20240826T221529_N0511_R115_T05WMN_20240827T001022 (2024-08-26)
   AOI coverage: 0.00%
   ⚠ Skipped

→ S2A_MSIL2A_20240825T215531_N0511_R029_T05WMN_20240826T024555 (2024-08-25)
   AOI coverage: 100.00%

→ S2B_MSIL2A_20240806T221529_N0511_R115_T05WMN_20240807T003953 (2024-08-06)
   AOI coverage: 0.00%
   ⚠ Skipped

→ S2B_MSIL2A_20240731T215529_N0511_R029_T05WMN_20240731T224931 (2024-07-31)
   AOI coverage: 100.00%

→ S2B_MSIL2A_20240724T220529_N0511_R072_T05WMN_20240725T002600 (2024-07-24)
   AOI coverage: 100.00%

→ S2A_MSIL2A_20240723T214531_N0511_R129_T05WMN_20240724T024358 (2024-07-23)
   AOI coverage: 0.00%
   ⚠ Skipped

→ S2A_MSIL

ERROR 1: PROJ: internal_proj_create_from_database: /home/pd/sipohl001/miniconda3/envs/env_omnicloud/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 3 is expected. It comes from another PROJ installation.


💾 Saving yearly median → CDSE_yearly_medians/median_2024.tif

==== YEAR 2025 | MGRS-05WMN | clouds ≤ 70% ====
  🔎 Found 27 items

→ S2C_MSIL2A_20250830T215551_N0511_R029_T05WMN_20250831T005013 (2025-08-30)
   AOI coverage: 10.75%
   ⚠ Skipped

→ S2C_MSIL2A_20250830T215551_N0511_R029_T05WMN_20250831T004359 (2025-08-30)
   AOI coverage: 100.00%

→ S2C_MSIL2A_20250820T215551_N0511_R029_T05WMN_20250821T005115 (2025-08-20)
   AOI coverage: 100.00%

→ S2B_MSIL2A_20250818T220529_N0511_R072_T05WMN_20250819T000337 (2025-08-18)
   AOI coverage: 100.00%

→ S2C_MSIL2A_20250817T214551_N0511_R129_T05WMN_20250818T022914 (2025-08-17)
   AOI coverage: 0.00%
   ⚠ Skipped

→ S2C_MSIL2A_20250816T221551_N0511_R115_T05WMN_20250817T010815 (2025-08-16)
   AOI coverage: 0.00%
   ⚠ Skipped

→ S2C_MSIL2A_20250807T214551_N0511_R129_T05WMN_20250808T002620 (2025-08-07)
   AOI coverage: 0.00%
   ⚠ Skipped

→ S2C_MSIL2A_20250806T221551_N0511_R115_T05WMN_20250806T234930 (2025-08-06)
   AOI coverage: 0.00%
   ⚠ Skipped

## omc (downloading masked images per year)

In [ ]:
# =========================================
# CONFIG
# =========================================
YEARS            = list(range(2017, 2021))
MONTH_START_END  = ("07-01", "08-31")
GRID             = "MGRS-05WMU"
MAX_CLOUD_COVER  = 70
BBOX_LL          = (-153.5, 70.5, -153, 71)
OUT_DIR          = "CDSE_scenes_masked/coverage70"

import os
os.environ["AWS_ACCESS_KEY_ID"]     = "C364NPCJK6JQ64OIMZJR"
os.environ["AWS_SECRET_ACCESS_KEY"] = "..."   # fill in securely
os.environ["AWS_REGION"]            = "us-east-1"
os.environ["AWS_S3_ENDPOINT"]       = "eodata.dataspace.copernicus.eu"
os.environ["AWS_VIRTUAL_HOSTING"]   = "FALSE"

# =========================================
# IMPORTS
# =========================================
from pathlib import Path
import math
import numpy as np
import xarray as xr
import rioxarray
from pyproj import Transformer
import rasterio as rio
from rasterio.enums import Resampling
from pystac_client import Client
from pystac import Item
from omnicloudmask import predict_from_array
from shapely.geometry import shape, box, mapping
from shapely.ops import transform as shapely_transform
# =========================================
# HELPERS
# =========================================
def search_s2_stac(start_date: str, end_date: str, grid: str, max_cloud_cover: int = 100) -> list[Item]:
    cat = Client.open("https://stac.dataspace.copernicus.eu/v1/")
    search = cat.search(
        collections=["sentinel-2-l2a"],
        datetime=f"{start_date}/{end_date}",
        query={"eo:cloud_cover": {"lte": max_cloud_cover}, "grid:code": {"eq": grid}},
    )
    items = list(search.items())
    print(f"  🔎 Found {len(items)} items")
    return items

def prefer_s3_assets(items):
    out = []
    for it in items:
        it = it.clone()
        for a in it.assets.values():
            s3_href = None
            extra = (getattr(a, "extra_fields", None) or {})
            alt = extra.get("alternate") or extra.get("alternates")
            if isinstance(alt, dict):
                s3_href = (alt.get("s3") or alt.get("S3") or {}).get("href")
            elif isinstance(alt, list):
                for d in alt:
                    href = d.get("href")
                    if href and href.startswith("s3://"):
                        s3_href = href
                        break
            if s3_href:
                a.href = s3_href
        out.append(it)
    return out

def detect_epsg_and_bounds(items, bbox_ll_override=None):
    if not items:
        raise ValueError("No items")

    if bbox_ll_override is None:
        bbs = [it.bbox for it in items]
        minx = min(b[0] for b in bbs)
        miny = min(b[1] for b in bbs)
        maxx = max(b[2] for b in bbs)
        maxy = max(b[3] for b in bbs)
        bbox_ll = (minx, miny, maxx, maxy)
    else:
        bbox_ll = bbox_ll_override

    epsg = None
    for it in items:
        if "proj:epsg" in it.properties:
            epsg = int(it.properties["proj:epsg"])
            break
    if epsg is None:
        lon = (bbox_ll[0] + bbox_ll[2]) / 2.0
        lat = (bbox_ll[1] + bbox_ll[3]) / 2.0
        zone = int(math.floor((lon + 180) / 6) + 1)
        epsg = 32600 + zone if lat >= 0 else 32700 + zone

    tx = Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)
    x1, y1 = tx.transform(bbox_ll[0], bbox_ll[1])
    x2, y2 = tx.transform(bbox_ll[2], bbox_ll[3])
    bounds_proj = (min(x1, x2), min(y1, y2), max(x1, x2), max(y1, y2))
    return epsg, bbox_ll, bounds_proj

def projected_intersection_ratio(item_geom, aoi_bounds, epsg_out):
    # Transform AOI bbox (in lon/lat) to projected coords
    tx = Transformer.from_crs("EPSG:4326", f"EPSG:{epsg_out}", always_xy=True)
    aoi_proj = shapely_transform(tx.transform, box(*aoi_bounds))

    # Get item's footprint and project it too
    geom = shape(item_geom)
    geom_proj = shapely_transform(tx.transform, geom)

    inter = geom_proj.intersection(aoi_proj)

    if inter.is_empty:
        return 0.0

    return inter.area / aoi_proj.area

def rasterio_env():
    return rio.Env(
        AWS_S3_ENDPOINT=os.environ["AWS_S3_ENDPOINT"],
        AWS_REGION=os.environ["AWS_REGION"],
        AWS_VIRTUAL_HOSTING=os.environ["AWS_VIRTUAL_HOSTING"],
        GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
        CPL_VSIL_CURL_ALLOWED_EXTENSIONS="tif,gtiff,jp2,xml"
    )

# =========================================
# PROCESS
# =========================================
BAND_ORDER  = ["B02_10m","B03_10m","B04_10m","B08_10m","B11_20m","B12_20m"]
BAND_LABELS = ["Blue","Green","Red","NIR","SWIR1","SWIR2"]

def process_year(year: int, grid: str, max_cloud: int, bbox_ll):
    print(f"\n==== Year {year} | grid={grid} | clouds≤{max_cloud}% ====")
    start_date = f"{year}-{MONTH_START_END[0]}"
    end_date   = f"{year}-{MONTH_START_END[1]}"

    items = search_s2_stac(start_date, end_date, grid, max_cloud_cover=max_cloud)
    if not items:
        print("  ⚠️ No items for this year.")
        return 0

    items_s3 = prefer_s3_assets(items)
    epsg_out, bbox_ll_used, bounds_out = detect_epsg_and_bounds(items, bbox_ll_override=bbox_ll)
    print(f"  EPSG={epsg_out} | bounds_proj={tuple(round(v,2) for v in bounds_out)}")

    bands_10m = [b for b in BAND_ORDER if b.endswith("10m")]
    bands_20m = [b for b in BAND_ORDER if b.endswith("20m")]

    out_dir = Path(OUT_DIR) / str(year)
    out_dir.mkdir(parents=True, exist_ok=True)

    n_ok = 0

    with rasterio_env():
        for it_s3, it_orig in zip(items_s3, items):
            scene_date = it_orig.properties.get("datetime", "").split("T")[0]
            print(f"\n→ Scene {it_orig.id} ({scene_date})")
            # ------------------------------------------------
            # AOI intersection check (BEFORE loading bands)
            # ------------------------------------------------
            coverage_ratio = projected_intersection_ratio(
                item_geom = it_orig.geometry,
                aoi_bounds = bbox_ll,      # in lon/lat!
                epsg_out = epsg_out        # detected for the tile
            )

            print(f"   ℹ️ AOI intersection coverage: {coverage_ratio:.2%}")

            if coverage_ratio < 0.4:  # Example: require 5% coverage
                print("   ⚠️ Scene skipped due to low AOI coverage.")
                continue

            try:
                ref = None
                pieces = []

                for bname in bands_10m:
                    if bname not in it_s3.assets:
                        continue
                    href = it_s3.assets[bname].href
                    da = rioxarray.open_rasterio(href, masked=True).squeeze("band", drop=True)
                    if da.rio.crs is None:
                        da = da.rio.write_crs(f"EPSG:{epsg_out}")
                    da = da.rio.clip_box(*bounds_out)
                    if ref is None:
                        ref = da
                    pieces.append(da.expand_dims("band"))

                for bname in bands_20m:
                    if bname not in it_s3.assets:
                        continue
                    href = it_s3.assets[bname].href
                    da20 = rioxarray.open_rasterio(href, masked=True).squeeze("band", drop=True)
                    if da20.rio.crs is None:
                        da20 = da20.rio.write_crs(f"EPSG:{epsg_out}")
                    da20 = da20.rio.clip_box(*bounds_out)
                    da20u = da20.rio.reproject_match(ref, resampling=Resampling.bilinear)
                    pieces.append(da20u.expand_dims("band"))

                if not pieces:
                    print("   ⚠️ No usable bands.")
                    continue

                scene = xr.concat(pieces, dim="band")
                scene = scene.assign_coords(band=BAND_LABELS)
                if scene.rio.crs is None:
                    scene = scene.rio.write_crs(f"EPSG:{epsg_out}")

                # MASKING
                red   = scene.sel(band="Red").values
                green = scene.sel(band="Green").values
                nir   = scene.sel(band="NIR").values
                input_array = np.stack([red, green, nir], axis=0)

                try:
                    pred_mask = predict_from_array(input_array)

                    # Handle shape (1, H, W) or (3, H, W)
                    if pred_mask.ndim == 3:
                        if pred_mask.shape[0] == 1:
                            pred_mask = pred_mask[0]
                        elif pred_mask.shape[0] == 3:
                            pred_mask = pred_mask[1]  # assume class 1 = cloud

                    # Ensure mask shape matches (y, x)
                    if pred_mask.shape != (scene.sizes["y"], scene.sizes["x"]):
                        raise ValueError(f"❌ Mask shape {pred_mask.shape} does not match scene shape {(scene.sizes['y'], scene.sizes['x'])}")

                    # Keep only pixels where class == 0
                    mask_keep = pred_mask == 0

                    mask_da = xr.DataArray(
                        mask_keep,
                        dims=("y", "x"),
                        coords={"y": scene.coords["y"], "x": scene.coords["x"]}
                    )

                    scene = scene.where(mask_da)
                    print("   ✔ Cloud mask applied.")

                except Exception as e:
                    print("   ⚠️ Cloud mask failed:", e)

                scene_u16 = (
                    scene.fillna(0)
                    .clip(0, 10000)
                    .astype("uint16")
                    .rio.write_nodata(0)
                    .rio.write_crs(f"EPSG:{epsg_out}")
                )

                out_path = out_dir / f"{it_orig.id}_{scene_date}_masked.tif"
                print("   💾 Saving →", out_path)
                scene_u16.transpose("band", "y", "x").rio.to_raster(
                    out_path,
                    driver="GTiff",
                    compress="deflate",
                    tiled=True,
                    predictor=2,
                    BIGTIFF="IF_SAFER",
                    blockxsize=512,
                    blockysize=512,
                )
                n_ok += 1

            except Exception as e:
                print("   ❌ Scene failed:", e)

    return n_ok

# =========================================
# RUN ALL YEARS
# =========================================
all_counts = {}
for yr in YEARS:
    n = process_year(yr, GRID, MAX_CLOUD_COVER, bbox_ll=BBOX_LL)
    all_counts[yr] = n

print("\n✅ Done. Scenes written per year:")
for yr, n in all_counts.items():
    print(f" • {yr}: {n} scenes")


## Stack to median

In [ ]:
from pathlib import Path
import xarray as xr
import rioxarray
import numpy as np

# --------------------------------------------
# CONFIG
# --------------------------------------------
TIF_DIR     = Path("CDSE_scenes_masked/coverage70/2025")  # Folder with the TIFFs
OUT_PATH    = Path("CDSE_2025_median_70.tif")   # Output file
BAND_LABELS = ["Blue", "Green", "Red", "NIR", "SWIR1", "SWIR2"]  # Optional

# --------------------------------------------
# LOAD TIFFS
# --------------------------------------------
tif_files = sorted(TIF_DIR.glob("*.tif"))
print(f"🗂 Found {len(tif_files)} TIFFs")

scenes = []

for f in tif_files:
    try:
        ds = rioxarray.open_rasterio(f, masked=True)  # shape: (band, y, x)
        if "band" not in ds.coords:
            ds = ds.assign_coords(band=range(1, ds.sizes["band"] + 1))

        # Optionally set band names
        if len(BAND_LABELS) == ds.sizes["band"]:
            ds = ds.assign_coords(band=BAND_LABELS)

        scenes.append(ds.expand_dims(time=[f.name]))  # add time dimension
        print(f"   ✓ Loaded {f.name}")
    except Exception as e:
        print(f"   ⚠️ Failed to load {f.name}: {e}")

if not scenes:
    raise RuntimeError("❌ No scenes could be loaded.")

# --------------------------------------------
# STACK + MEDIAN
# --------------------------------------------
stack = xr.concat(scenes, dim="time")
print("📊 Stack shape:", stack.shape)

median_img = stack.median(dim="time", skipna=True)

# --------------------------------------------
# SAVE MEDIAN STACK
# --------------------------------------------
median_img_u16 = (
    median_img
    .clip(0, 10000)
    .fillna(0)
    .astype("uint16")
    .rio.write_nodata(0)
)

print(f"💾 Saving median image to {OUT_PATH}")
median_img_u16.rio.to_raster(
    OUT_PATH,
    driver="GTiff",
    compress="deflate",
    tiled=True,
    predictor=2,
    BIGTIFF="IF_SAFER",
    blockxsize=512,
    blockysize=512,
)

print("✅ Done.")


## Calculating TC images

In [2]:
# === tasseled_cap_mosaic_generation.py ===
from pathlib import Path
import numpy as np
import xarray as xr
import rioxarray
from dask.diagnostics import ProgressBar
import warnings

warnings.filterwarnings("ignore", category=UserWarning, message=".*coordinate precision.*")

median_dir = Path("CDSE_yearly_medians")    #("omc_medians_70")       
tc_dir = Path("omc_tc_70_south")                      
tc_dir.mkdir(exist_ok=True)
years = list(range(2017, 2026))

# Sentinel-2 Tasseled Cap coefficients 
coeffs = {
    "tcb": dict(Blue=0.3037, Green=0.2793, Red=0.4743, NIR=0.5585, SWIR1=0.5082, SWIR2=0.1863),
    "tcg": dict(Blue=-0.2848, Green=-0.2435, Red=-0.5436, NIR=0.7243, SWIR1=0.0840, SWIR2=-0.1800),
    "tcw": dict(Blue=0.1509, Green=0.1973, Red=0.3279, NIR=0.3406, SWIR1=-0.7112, SWIR2=-0.4572),
}

for year in years:
    in_file = median_dir / f"median_{year}.tif"
    out_file = tc_dir / f"median_{year}_tc_omc_70_south.tif"

    if not in_file.exists():
        print(f"❌ Missing median mosaic for {year}")
        continue
    if out_file.exists():
        print(f"⏭️ Already exists, skipping {out_file}")
        continue

    print(f"✅ Loading: {in_file}")
    # Important: masked=True makes rioxarray treat nodata (0) as NaN
    da = rioxarray.open_rasterio(in_file, chunks={"x": 1024, "y": 1024}, masked=True)

    # assign band names, convert reflectance to 0–1
    da = da.assign_coords(band=["Blue", "Green", "Red", "NIR", "SWIR1", "SWIR2"]).astype("float32") / 10000.0          # not needed - 0.1 

    # Ensure true zeros are NaN (in case old medians used fillna(0))
    da = da.where(da != 0)

    # Split bands
    blue, green, red, nir, swir1, swir2 = da.sel(band=["Blue", "Green", "Red", "NIR", "SWIR1", "SWIR2"])

    def tc(c):
        return (c["Blue"]*blue + c["Green"]*green + c["Red"]*red +
                c["NIR"]*nir + c["SWIR1"]*swir1 + c["SWIR2"]*swir2)

    # Compute tasseled cap 
    tcb = tc(coeffs["tcb"])
    tcg = tc(coeffs["tcg"])
    tcw = tc(coeffs["tcw"])

    # Stack tc
    tc_stack = xr.concat([tcb, tcg, tcw], dim="band")
    tc_stack = tc_stack.assign_coords(band=["TCB", "TCG", "TCW"])
    tc_stack = tc_stack.rio.write_crs(da.rio.crs)

    # Ensure NaNs are preserved
    tc_stack = tc_stack.astype("float32").rio.write_nodata(np.nan)

    print(f"💾 Saving tasseled cap mosaic: {out_file}")
    with ProgressBar():
        (
            tc_stack.compute(scheduler="threads")
            .transpose("band", "y", "x")
            .rio.to_raster(
                out_file,
                driver="GTiff",
                tiled=True,
                compress="deflate",
                BIGTIFF="IF_SAFER",
                predictor=3,           
                blockxsize=1024,
                blockysize=1024,
            )
        )

print("✅ All tasseled cap mosaics saved.")


✅ Loading: CDSE_yearly_medians/median_2017.tif
💾 Saving tasseled cap mosaic: omc_tc_70_south/median_2017_tc_omc_70_south.tif
[########################################] | 100% Completed | 1.82 sms
✅ Loading: CDSE_yearly_medians/median_2018.tif
💾 Saving tasseled cap mosaic: omc_tc_70_south/median_2018_tc_omc_70_south.tif
[########################################] | 100% Completed | 1.72 sms
✅ Loading: CDSE_yearly_medians/median_2019.tif
💾 Saving tasseled cap mosaic: omc_tc_70_south/median_2019_tc_omc_70_south.tif
[########################################] | 100% Completed | 2.02 sms
✅ Loading: CDSE_yearly_medians/median_2020.tif
💾 Saving tasseled cap mosaic: omc_tc_70_south/median_2020_tc_omc_70_south.tif
[########################################] | 100% Completed | 2.23 sms
✅ Loading: CDSE_yearly_medians/median_2021.tif
💾 Saving tasseled cap mosaic: omc_tc_70_south/median_2021_tc_omc_70_south.tif
[########################################] | 100% Completed | 2.02 sms
✅ Loading: CDSE_year

## Trend Calculation (fixed vis)

In [4]:
# =========================================
# TREND CALCULATION FOR TC STACKS
# =========================================

from pathlib import Path
import numpy as np
import xarray as xr
import rioxarray
from dask.diagnostics import ProgressBar
import dask
import logging

# -----------------------------------------
# CONFIG
# -----------------------------------------
tc_dir     = Path("omc_tc_70_south")          # input mosaics
trend_dir  = Path("omc_trends")  # output directory
trend_dir.mkdir(exist_ok=True)

years = list(range(2017, 2026))
bands_tc = ["TCB", "TCG", "TCW"]

# -----------------------------------------
# 1. LOAD ALL TASSELED CAP MOSAICS
# -----------------------------------------
arrays = []

for year in years:
    fp = tc_dir / f"median_{year}_tc_omc_70_south.tif"
    if not fp.exists():
        print(f"❌ Missing {fp}")
        continue

    print(f"✅ Loading {fp}")
    da = rioxarray.open_rasterio(fp, chunks={"x": 1024, "y": 1024})

    # Assign TC band names
    da = da.assign_coords(band=bands_tc)

    # Add numeric time coordinate
    da = da.expand_dims(time=[np.datetime64(f"{year}-07-15")])

    arrays.append(da)

if not arrays:
    raise RuntimeError("No tasseled cap mosaics found!")

# Concatenate stack
stack = xr.concat(arrays, dim="time").transpose("time", "band", "y", "x")
stack = stack.chunk({"time": -1, "x": 1024, "y": 1024})
stack.name = "tc"

print(f"🧩 Stack shape: {stack.shape} (time, band, y, x)")

# -----------------------------------------
# 2. FIX THE TIME AXIS FOR REGRESSION
# -----------------------------------------
# Convert datetime64 → integer years
years_numeric = stack["time"].dt.year

# Replace time dim with 'year'
stack = stack.assign_coords(year=("time", years_numeric.data))
stack = stack.swap_dims({"time": "year"})

print(f"📅 Using year values for regression: {list(years_numeric.values)}")

# -----------------------------------------
# 3. TREND REGRESSION (PER YEAR)
# -----------------------------------------
results = []

for band in bands_tc:
    print(f"📈 Computing trend for {band}...")

    sub = stack.sel(band=band)

    # Fit a first-degree polynomial across the 'year' axis
    fit = sub.to_dataset(name="tc").polyfit(dim="year", deg=1)

    # Extract slope (degree 1 coefficient)
    slope = fit["tc_polyfit_coefficients"].sel(degree=1)

    # OPTIONAL —
    # match GEE visualization intensity (your GEE script did "*10")
    slope = slope * 10

    slope = slope.expand_dims(band=[f"{band}_slope"])
    results.append(slope)

# Combine all slope bands
trend = xr.concat(results, dim="band")
trend.rio.write_crs(stack.rio.crs, inplace=True)

# -----------------------------------------
# 4. COMPUTE THE ARRAY
# -----------------------------------------
out_path = trend_dir / "tc_trend_omc_south.tif"
print(f"💾 Saving trend raster: {out_path}")

# Threaded Dask scheduler
dask.config.set(scheduler="threads")
logging.getLogger("tornado.application").setLevel(logging.ERROR)
logging.getLogger("tornado.general").setLevel(logging.ERROR)

with ProgressBar(dt=30.0):  
    trend = trend.compute()

trend_vis = trend.clip(-0.3, 0.3)
trend_vis = ((trend_vis + 0.3) / 0.6 * 255).astype("uint8")
trend_vis.transpose("band", "y", "x").rio.to_raster("trend_visual_70_no2024.tif")

# -----------------------------------------
# 5. SAVE TO GEOTIFF
# -----------------------------------------
trend_vis.transpose("band", "y", "x").rio.to_raster(
    out_path,
    driver="GTiff",
    tiled=True,
    compress="deflate",
    BIGTIFF="IF_SAFER",
    predictor=2,
    blockxsize=1024,
    blockysize=1024,
)

print("✅ Trend image saved successfully.")



✅ Loading omc_tc_70_south/median_2017_tc_omc_70_south.tif
✅ Loading omc_tc_70_south/median_2018_tc_omc_70_south.tif
✅ Loading omc_tc_70_south/median_2019_tc_omc_70_south.tif
✅ Loading omc_tc_70_south/median_2020_tc_omc_70_south.tif
✅ Loading omc_tc_70_south/median_2021_tc_omc_70_south.tif
✅ Loading omc_tc_70_south/median_2022_tc_omc_70_south.tif
✅ Loading omc_tc_70_south/median_2023_tc_omc_70_south.tif
✅ Loading omc_tc_70_south/median_2024_tc_omc_70_south.tif
✅ Loading omc_tc_70_south/median_2025_tc_omc_70_south.tif
🧩 Stack shape: (9, 3, 2766, 1224) (time, band, y, x)
📅 Using year values for regression: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
📈 Computing trend for TCB...
📈 Computing trend for TCG...
📈 Computing trend for TCW...
💾 Saving trend raster: omc_trends/tc_trend_omc_south.tif
[############################            ] | 70% Completed | 153.63 s

IOStream.flush timed out


[#####################################   ] | 92% Completed | 281.95 s

IOStream.flush timed out
IOStream.flush timed out


[########################################] | 100% Completed | 339.60 s
✅ Trend image saved successfully.
